# 02 — Walk-Forward Validation of the Multivariate Pair Trading Strategy

End-to-end evaluation of the MPTS with **expanding-window walk-forward
validation** — every fold re-calibrates the signal thresholds,
mean-reversion speed, expected returns and pair covariances on its own
train window, then trades the next 12 months out-of-sample:

1. illustrate the threshold grid search on the first train window;
2. run the walk-forward loop;
3. per-fold diagnostics: chosen thresholds, trade counts, Sharpe dispersion;
4. stitched out-of-sample equity curve vs. buy-and-hold benchmarks.

A single 70/30 split was used in the original report; it calibrated the
thresholds once, on one market regime, and judged them on another — the
walk-forward structure replaces it.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml

from src import backtest, data, hedging, metrics, signals, stat_tests, walk_forward

cfg = yaml.safe_load(open("../configs/bloomberg.yaml", encoding="utf-8")) # select default.yaml if Bloomberg data is not available
ANCHOR = cfg["universe"]["anchor"]
EQUITIES = cfg["universe"]["equities"]
WINDOW = cfg["signals"]["zscore_window"]
WF = cfg["walk_forward"]
RF = cfg["strategy"]["risk_free_rate"]
TC = cfg["strategy"]["transaction_cost"]

In [ ]:
prices = data.load_prices("../" + cfg["data"]["path"])
log_prices = data.to_log_prices(prices)

## 1. Threshold calibration — what one fold sees

Grid search over `open ∈ [1.0, 2.5)` and `close ∈ [0.5, 2.0)` (step 0.1,
`close < open` enforced) on the **first train window**, maximizing the
average single-spread Sharpe across the basket. The walk-forward loop
repeats this per fold and picks the *median of the top decile* (the plateau)
instead of the argmax cell, reducing per-fold selection bias.

In [ ]:
log_screen = data.screening_window(log_prices, WF["min_train_years"])
grid_results, heatmap = signals.grid_search_thresholds(log_screen, EQUITIES, ANCHOR, window=WINDOW, risk_free_rate=RF)
open_grid = np.round(np.arange(1.0, 2.5, 0.1), 1)
close_grid = np.round(np.arange(0.5, 2.0, 0.1), 1)

fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(heatmap, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=close_grid, yticklabels=open_grid,
            linewidths=0.5, cbar_kws={"label": "avg Sharpe"}, ax=ax)
ax.set_xlabel("Closing threshold")
ax.set_ylabel("Opening threshold")
plt.tight_layout()
plt.show()

plateau_open, plateau_close = walk_forward.calibrate_thresholds(
    log_screen, EQUITIES, ANCHOR, window=WINDOW, risk_free_rate=RF
)
print(f"Plateau thresholds for fold 1: open=\u00b1{plateau_open}, close=\u00b1{plateau_close}")

## 2. Run the walk-forward validation

In [ ]:
result = walk_forward.run_walk_forward(
    log_prices, EQUITIES, ANCHOR, [ANCHOR, cfg["universe"]["sector_proxy"]],
    min_train_years=WF["min_train_years"],
    test_months=WF["test_months"],
    embargo_days=WF["embargo_days"],
    window=WINDOW,
    risk_aversion=cfg["strategy"]["risk_aversion"],
    transaction_cost=TC,
    risk_free_rate=RF,
    rolling_beta_window=cfg["strategy"]["rolling_beta_window"],
    solver=cfg["strategy"]["solver"],
)
result.folds.round(3)

The per-fold table is the honest headline: with ~6 annual folds, report the
**median fold Sharpe and its range**, not a single number. Note how the
chosen thresholds move across folds — threshold economics are
regime-dependent, which the single-split design could never reveal.

## 3. Stitched out-of-sample equity curve vs. benchmarks

In [ ]:
CAPITAL = cfg["backtest"]["initial_capital"]
log_oos = log_prices.loc[result.pnl.index]
kc1_value = backtest.buy_and_hold_benchmark(log_oos, [ANCHOR], CAPITAL, TC)
ew_value = backtest.buy_and_hold_benchmark(log_oos, EQUITIES, CAPITAL, TC)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(result.portfolio_value(CAPITAL), lw=1.4,
        label=f"MPTS walk-forward (Sharpe {result.sharpe:.2f})")
ax.plot(kc1_value, ls=":", color="saddlebrown", lw=1.3, label=f"B&H {ANCHOR}")
ax.plot(ew_value, ls=":", color="grey", lw=1.3, label="B&H EW basket")
for test_start in pd.to_datetime(result.folds["test_start"]).iloc[1:]:
    ax.axvline(test_start, color="grey", ls="--", lw=0.7, alpha=0.6)
ax.set_ylabel("Portfolio value ($)")
ax.legend()
ax.grid(alpha=0.4)
plt.tight_layout()
plt.show()

report = pd.DataFrame({
    "MPTS walk-forward": metrics.portfolio_metrics(result.portfolio_value(CAPITAL), RF),
    f"B&H {ANCHOR}": metrics.portfolio_metrics(kc1_value, RF),
    "B&H EW basket": metrics.portfolio_metrics(ew_value, RF),
})
report.round(4)

## 4. Trade-level statistics and fold dispersion

In [ ]:
display(pd.Series(metrics.trade_metrics(result.trades)).round(4).to_frame("value"))
print(f"Fold Sharpe: median {result.folds['sharpe'].median():.3f}, "
      f"range [{result.folds['sharpe'].min():.3f}, {result.folds['sharpe'].max():.3f}]")

## Takeaways

* The walk-forward structure replaces the single 70/30 split: thresholds are
  re-calibrated on every expanding train window and only ever judged on
  unseen test months, with a 1-month embargo between the two.
* Fold-level dispersion is the honest measure of robustness — a strategy
  whose Sharpe swings sign across folds is regime-dependent regardless of
  how good the aggregate number looks.
* The structural limitation remains statistical, not methodological: the
  assigned equity universe shows little cointegration with the coffee
  future, so the mean-reversion premise is weak. Trading the
  Arabica/Robusta (`KC1`/`DF1`) inter-commodity spread would be the natural
  cointegrated alternative — noted as future work.